In [1]:
import pandas as pd

running = pd.read_csv("../data/processed/bigwest_2021_2026_running.csv")
field = pd.read_csv("../data/processed/bigwest_2021_2026_field.csv")

athletes = pd.concat([
    running[["athlete_id", "athlete_name"]],
    field[["athlete_id", "athlete_name"]]
]).drop_duplicates(subset=["athlete_id"])

In [2]:
athlete_ids = (
    athletes["athlete_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda x: x.ne("")]
    .unique()
)

If you want to scrape from tffrs page use api key from https://parse.bot

In [3]:
url = "https://api.parse.bot/scraper/1b66c7cb-769a-4301-82ff-8d308afee7ad/get_athlete_results"

headers = {
    "X-API-Key": "$API_key",
    "API-Snapshot-Version": "5"
}

In [4]:
import requests
import pandas as pd
import time

# Load existing results
results_df = pd.read_csv("../data/raw/bigwest_regular_season_partial.csv")

# Track completed athletes
completed_ids = set(
    results_df["athlete_id"].dropna().astype(str).str.strip()
)

# Find athletes still needing data
remaining_ids = [
    str(x).strip()
    for x in athlete_ids
    if str(x).strip() not in completed_ids
]

remaining_ids = list(dict.fromkeys(remaining_ids))

print(f"Starting collection for {len(remaining_ids)} athletes")

new_rows = []

for i, athlete_id in enumerate(remaining_ids):

    try:
        response = requests.get(
            url,
            headers=headers,
            params={"athlete_id": athlete_id},
            timeout=120
        )

        # Stop when rate-limited
        if response.status_code == 429:
            print("Rate limit reached.")
            print("Retry after:", response.headers.get("Retry-After"))
            print("Daily remaining:", response.headers.get(
                "x-ratelimit-daily-remaining"
            ))
            break

        response.raise_for_status()

        result = response.json()

        if result.get("status") != "success":
            print(f"Unsuccessful response for {athlete_id}")
            continue

        athlete = result["data"]

        athlete_rows = []

        for performance in athlete.get("performances", []):

            meet_date = str(performance.get("meet_date", ""))

            # Remove incorrectly parsed records
            if not any(month in meet_date for month in [
                "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
            ]):
                continue

            athlete_rows.append({
                "athlete_id": athlete_id,
                "athlete_name": athlete.get("name"),
                "team": athlete.get("team"),
                **performance
            })

        if athlete_rows:
            new_rows.extend(athlete_rows)

            # Combine old and new results
            updated_df = pd.concat(
                [results_df, pd.DataFrame(new_rows)],
                ignore_index=True
            )

            updated_df = updated_df.drop_duplicates()

            # Save after each athlete
            updated_df.to_csv(
                "../data/raw/bigwest_regular_season_partial.csv",
                index=False
            )

        completed_ids.add(athlete_id)

        print(
            f"Completed {i + 1}/{len(remaining_ids)}: "
            f"{athlete.get('name')}"
        )

        # Stop near daily quota
        daily_remaining = response.headers.get(
            "x-ratelimit-daily-remaining"
        )

        if daily_remaining is not None and int(daily_remaining) <= 1:
            print("Daily API quota almost exhausted.")
            break

    except (requests.RequestException, ValueError, KeyError, TypeError) as e:
        print(f"Failed {athlete_id}: {e}")

    # Stay within the 5 requests/minute refill rate
    time.sleep(4)

print("Collection stopped. Progress saved.")

Starting collection for 0 athletes
Collection stopped. Progress saved.


In [5]:
results_df = pd.read_csv("../data/raw/bigwest_regular_season_partial.csv")

completed_ids = set(
    results_df["athlete_id"].dropna().astype(str).str.strip()
)

remaining_ids = set(
    str(x).strip() for x in athlete_ids
) - completed_ids

print("Total athletes:", len(athlete_ids))
print("Athletes collected:", len(completed_ids))
print("Athletes remaining:", len(remaining_ids))
print("Total performance records:", len(results_df))
print(results_df.shape)

Total athletes: 1514
Athletes collected: 1514
Athletes remaining: 0
Total performance records: 64580
(64580, 11)


## Cleaning Data

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/bigwest_regular_season_partial.csv")

# Rename columns
df = df.rename(columns={
    "athete_name": "athlete_name",
    "is_track_field": "is_track"
})

# Clean whitespace
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = (
        df[col].astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

# Preserve original marks for extraction
original_mark = df["mark"].astype("string")

# Extract year, including dates like "Apr 13-16, 2022"
df["year"] = pd.to_numeric(
    df["meet_date"].astype("string").str.extract(r"\b(\d{4})\b")[0],
    errors="coerce"
).astype("Int64")

# Extract meters from marks like 6.72m(0.2)22' 0.75"
meters = pd.to_numeric(
    original_mark.str.extract(r"(\d+(?:\.\d+)?)\s*m\b")[0],
    errors="coerce"
)

# Extract wind
df["wind"] = pd.to_numeric(
    original_mark.str.extract(
        r"^\s*\d+(?:\.\d+)?\s*m?\s*\(\s*([+-]?\d+(?:\.\d+)?)\s*\)"
    )[0],
    errors="coerce"
)

# Read running times in seconds
seconds = pd.to_numeric(df["mark_seconds"], errors="coerce")

# Match event labels without changing the original event column
event = (
    df["event"].astype("string")
    .str.upper()
    .str.replace(r"\s+", "", regex=True)
)

field_event = event.isin([
    "LJ", "TJ", "HJ", "PV", "SP", "DT", "HT", "JT", "WT"
])

running_event = event.isin([
    "60", "100", "200", "400", "800", "1000", "1500",
    "MILE", "3000", "5000", "10000",
    "60H", "100H", "110H", "400H",
    "2000S", "3000S", "4X100", "4X400", "4X800", "DMR","10,000"
])

points_event = event.isin([
    "DEC", "HEP", "PENT", "DECATHLON", "HEPTATHLON", "PENTATHLON","Dec","Hep"
])

# Combine performances into one mark column
df["mark"] = original_mark.str.replace(
    r"\(\s*[+-]?\d+(?:\.\d+)?\s*\)", "", regex=True
).str.strip()
df["mark_unit"] = pd.Series(pd.NA, index=df.index, dtype="string")

df.loc[seconds.notna() | running_event, "mark_unit"] = "time"

df.loc[meters.notna(), "mark"] = meters[meters.notna()].astype("string")
df.loc[meters.notna() | field_event, "mark_unit"] = "meters"

# Combined events use points
points = pd.to_numeric(
    original_mark.str.replace(",", "", regex=False),
    errors="coerce"
)
df.loc[points_event, "mark"] = points[points_event].astype("string")
df.loc[points_event, "mark_unit"] = "points"

# Preserve non-performance statuses
status = original_mark.str.strip().str.upper()
is_status = status.isin(["DNF", "DQ", "DNS", "NM", "NH", "FOUL"])
df.loc[is_status, "mark"] = status[is_status]

# Extract numeric placement
df["place"] = pd.to_numeric(
    df["place"].astype("string").str.extract(r"^(\d+)")[0],
    errors="coerce"
).astype("Int64")

# Select final columns
df = df[[
    "athlete_id",
    "athlete_name",
    "team",
    "meet_name",
    "meet_date",
    "year",
    "event",
    "mark",
    "mark_unit",
    "place",
    "is_track",
    "wind"
]].copy()

df = df[df["is_track"] == True].copy()

df["event_type"] = "unknown"

df.loc[running_event.reindex(df.index, fill_value=False), "event_type"] = "track"
df.loc[field_event.reindex(df.index, fill_value=False), "event_type"] = "field"
df.loc[points_event.reindex(df.index, fill_value=False), "event_type"] = "combined"
df.head()

,athlete_id,athlete_name,team,meet_name,meet_date,year,event,mark,mark_unit,place,is_track,wind,event_type
0,7657050,Benjamin Goodwin,UC DAVIS,62nd Annual Mt. SAC Relays (Collegiate),"Apr 13-16, 2022",2022,100,10.74,time,58,True,1.9,track
1,7657050,Benjamin Goodwin,UC DAVIS,62nd Annual Mt. SAC Relays (Collegiate),"Apr 13-16, 2022",2022,4x100,41.06,time,4,True,<NA>,track
2,7657050,Benjamin Goodwin,UC DAVIS,Brutus Hamilton Invitational,"Apr 9, 2022",2022,100,10.95,time,10,True,-1.1,track
3,7657050,Benjamin Goodwin,UC DAVIS,Rafer Johnson & Jackie Joyner- Kersee Invite,"Mar 26, 2022",2022,100,10.88,time,12,True,-0.4,track
4,7657050,Benjamin Goodwin,UC DAVIS,Rafer Johnson & Jackie Joyner- Kersee Invite,"Mar 26, 2022",2022,4x100,40.51,time,1,True,<NA>,track


## Code used to fix the team column so its only bigwest schools and not school from transferred


In [7]:
roster_df = pd.read_csv("../data/processed/bigwest_roster.csv")
results_df = df.copy()
# Standardize athlete IDs and years
for data in [roster_df, results_df]:
    data["athlete_id"] = data["athlete_id"].astype("string").str.strip()
    data["year"] = pd.to_numeric(data["year"], errors="coerce").astype("Int64")

# Keep only the columns needed for matching
roster_lookup = roster_df[
    ["athlete_id", "year", "school"]
].drop_duplicates()

roster_lookup.head()
roster_lookup = roster_lookup.drop(index=3093)
roster_lookup = roster_lookup.drop(index=3222)

In [8]:
duplicates = roster_lookup[
    roster_lookup.duplicated(
        subset=["athlete_id", "year"],
        keep=False
    )
]

print("Conflicting roster records:", len(duplicates))

duplicates.sort_values(["athlete_id", "year"])

Conflicting roster records: 0


,athlete_id,year,school


In [9]:
results_updated = results_df.merge(
    roster_lookup,
    on=["athlete_id", "year"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_roster")
)

In [10]:
# Preserve original team for comparison
results_updated["original_team"] = results_updated["team"]

results_updated["team"] = results_updated["school"]
results_updated = results_updated.dropna(
    subset=["team"]
).copy()

# Remove temporary column
results_updated = results_updated.drop(columns=["school"])

In [11]:
changed = results_updated[
    results_updated["team"] != results_updated["original_team"]
]

print("Total performances:", len(results_updated))
print("Performances with corrected teams:", len(changed))
print("Unique athletes affected:", changed["athlete_id"].nunique())

changed[
    ["athlete_id", "year", "original_team", "team"]
].drop_duplicates().head(20)

Total performances: 49682
Performances with corrected teams: 2697
Unique athletes affected: 84


,athlete_id,year,original_team,team
41,7149403,2022,USC,UC DAVIS
62,7149403,2021,USC,UC DAVIS
785,7152237,2021,USC,CAL POLY
1201,7781421,2024,HOWARD,LONG BEACH ST.
1226,7781421,2022,HOWARD,LONG BEACH ST.
1257,7781421,2021,HOWARD,LONG BEACH ST.
1302,7520943,2023,LONG BEACH ST.,CAL ST. FULLERTON
1330,7520943,2022,LONG BEACH ST.,CAL ST. FULLERTON
1364,7520943,2021,LONG BEACH ST.,CAL ST. FULLERTON
2071,7780841,2022,LONG BEACH ST.,CSUN


In [12]:
roster_lookup.to_csv(
    "../data/processed/bigwest_roster.csv",
    index=False
)

In [13]:
##Clean so everything before 2019 is deleted
results_updated = results_updated[results_updated["year"]>=2021]

## Final cleaning make it so it actually represents regular season performances not evrythign after or before

In [14]:
big_west_schools = {
    "UC SAN DIEGO",
    "CSU BAKERSFIELD",
    "CSUN",
    "CAL ST. FULLERTON",
    "LONG BEACH ST.",
    "UC DAVIS",
    "UC IRVINE",
    "UC RIVERSIDE",
    "UC SANTA BARBARA",
    "CAL POLY",
    "HAWAII"
}

In [15]:
import pandas as pd
import re

def parse_meet_date(date_string):
    if pd.isna(date_string):
        return pd.NaT

    date_string = str(date_string).strip()

    # Extract first month, first day, and year
    match = re.match(
        r"^([A-Za-z]+)\s+(\d{1,2}).*?,\s*(\d{4})$",
        date_string
    )

    if match:
        month, day, year = match.groups()

        return pd.to_datetime(
            f"{month} {day}, {year}",
            errors="coerce"
        )

    return pd.to_datetime(date_string, errors="coerce")


# Apply the corrected function
df = results_updated.copy()
df["meet_start_date"] = df["meet_date"].apply(parse_meet_date)

# Check remaining failures
print(
    "Dates that couldn't be parsed:",
    df["meet_start_date"].isna().sum()
)

Dates that couldn't be parsed: 0


In [16]:
df = results_updated.copy()
df["meet_start_date"] = df["meet_date"].apply(parse_meet_date)
# Convert year to integer
df["year"] = pd.to_numeric(
    df["year"], errors="coerce"
).astype("Int64")

# Championship final dates (fill in verified dates for each year)
championship_end_dates = {
    2026: pd.Timestamp("2026-05-16"),
    2025: pd.Timestamp("2025-05-17"),
    2024: pd.Timestamp("2024-05-11"),
    2023: pd.Timestamp("2023-05-13"),
    2022: pd.Timestamp("2022-05-14"),
    2021: pd.Timestamp("2021-05-15"),
    
}

df["championship_end_date"] = df["year"].map(
    championship_end_dates
)

# Only Big West schools
valid_team = (
    df["team"].str.upper().str.strip().isin(big_west_schools)
)

# Only January through May
valid_month = df["meet_start_date"].dt.month.between(1, 5)

# Identify Big West meets
is_big_west = df["meet_name"].str.contains(
    r"big\s*west",
    case=False,
    na=False,
    regex=True
)

# Identify Big West Challenge Cup
is_challenge_cup = df["meet_name"].str.contains(
    r"big\s*west\s*challenge\s*cup",
    case=False,
    na=False,
    regex=True
)

# Remove Big West meets EXCEPT the Challenge Cup
not_big_west = ~is_big_west | is_challenge_cup


# Remove meets starting after championship ends
before_cutoff = (
    df["meet_start_date"] <= df["championship_end_date"]
)

# Apply filters
regular_season_clean = df[
    valid_team &
    valid_month &
    not_big_west &
    before_cutoff
].copy()

# Remove temporary column
regular_season_clean.drop(
    columns=["championship_end_date"],
    inplace=True
)

print("Original rows:", len(df))
print("Remaining rows:", len(regular_season_clean))
print("Removed rows:", len(df) - len(regular_season_clean))

Original rows: 49682
Remaining rows: 41274
Removed rows: 8408


In [17]:
regular_season_clean.to_csv(
    "../data/processed/regular_season.csv",
    index=False
)

In [18]:
deleted_rows = df.loc[
    ~df.index.isin(regular_season_clean.index)
].copy()

print("Total deleted rows:", len(deleted_rows))

deleted_rows

Total deleted rows: 8408


,athlete_id,athlete_name,team,meet_name,meet_date,year,event,mark,mark_unit,place,is_track,wind,event_type,original_team,meet_start_date,championship_end_date
8,7657050,Benjamin Goodwin,UC DAVIS,2021 Big West Outdoor Track & Field Championships,"May 12-15, 2021",2021,100,10.67,time,1,True,1.2,track,UC DAVIS,2021-05-12,2021-05-15
9,7657050,Benjamin Goodwin,UC DAVIS,2021 Big West Outdoor Track & Field Championships,"May 12-15, 2021",2021,100,10.78,time,2,True,-0.4,track,UC DAVIS,2021-05-12,2021-05-15
10,7657050,Benjamin Goodwin,UC DAVIS,2021 Big West Outdoor Track & Field Championships,"May 12-15, 2021",2021,200,21.87,time,6,True,0.4,track,UC DAVIS,2021-05-12,2021-05-15
11,7657050,Benjamin Goodwin,UC DAVIS,2021 Big West Outdoor Track & Field Championships,"May 12-15, 2021",2021,200,21.90,time,7,True,0.8,track,UC DAVIS,2021-05-12,2021-05-15
12,7657050,Benjamin Goodwin,UC DAVIS,2021 Big West Outdoor Track & Field Championships,"May 12-15, 2021",2021,4x100,41.05,time,2,True,<NA>,track,UC DAVIS,2021-05-12,2021-05-15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60212,9003873,Maya Holman,HAWAII,Big West Championships,"May 8-16, 2026",2026,200,26.17,time,13,True,1.2,track,HAWAII,2026-05-08,2026-05-16
60213,9003873,Maya Holman,HAWAII,Big West Championships,"May 8-16, 2026",2026,SP,"12.26m40' 2.75""",meters,3,True,<NA>,field,HAWAII,2026-05-08,2026-05-16
60214,9003873,Maya Holman,HAWAII,Big West Championships,"May 8-16, 2026",2026,HJ,"1.62m5' 3.75""",meters,4,True,<NA>,field,HAWAII,2026-05-08,2026-05-16
60215,9003873,Maya Holman,HAWAII,Big West Championships,"May 8-16, 2026",2026,100H,14.58,time,7,True,0.7,track,HAWAII,2026-05-08,2026-05-16
